In [ ]:
# Deep Learning - Swish activation function
# Date: 01.10.2025

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import cv2
import numpy as np
import os
from sklearn.utils import shuffle
from scipy.stats import skew
from scipy.fft import fft
import glob

# --- Helper Functions from PDF ---
def swish(x):
    """Swish activation function."""
    return x / (1 + np.exp(-x))

def swish_derivative(x):
    """Derivative of the Swish function."""
    sig_x = 1 / (1 + np.exp(-x))
    # Formula from PDF
    return swish(x) + sig_x * (1 - swish(x))

def extract_features(image_path):

    # --- B) Color Index Features (Based on my teacher's note) ---
    color_image = cv2.imread(image_path)
    if color_image is None:
        print(f"Warning: Could not read {image_path}.")
        return None

    # Use float64 for means to prevent overflow
    B_mean, G_mean, R_mean = np.mean(color_image, axis=(0, 1), dtype=np.float64)
    epsilon = 1e-6 # Prevent division by zero

    # 5 Color Index Features
    brightness = ((R_mean**2) + (G_mean**2) + (B_mean**2)) / 3
    saturation = (R_mean - B_mean) / (R_mean + B_mean + epsilon)
    hue = (2*R_mean - G_mean - B_mean) / (G_mean - B_mean + epsilon)
    coloration = (R_mean - G_mean) / (R_mean + G_mean + epsilon)
    redness = (R_mean**2) / (B_mean * (G_mean**3) + epsilon)

    color_features = [brightness, saturation, hue, coloration, redness]

    # --- A) Gradient Features ---
    gray_image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    gray_image_resized = cv2.resize(gray_image, (128, 128))

    # Ensure we use float64 in Sobel
    sobelx = cv2.Sobel(gray_image_resized, cv2.CV_64F, 1, 0, ksize=5)
    sobely = cv2.Sobel(gray_image_resized, cv2.CV_64F, 0, 1, ksize=5)
    gradient = np.abs(sobelx) + np.abs(sobely)
    vector = gradient.reshape(1, 16384)

    # 8 Gradient Features
    mean = np.mean(vector)
    std_dev = np.std(vector)
    skewness = skew(vector, axis=None)
    variance = np.var(vector)
    median = np.median(vector)
    percentile_50 = np.percentile(vector, 50)

    hist = np.histogram(vector, bins=256)[0]
    hist_norm = hist / hist.sum()
    entropy = -np.sum(hist_norm * np.log2(hist_norm + epsilon))

    fft_vals = np.abs(fft(vector))
    mean_fft = np.mean(fft_vals)

    gradient_features = [mean, entropy, std_dev, skewness, variance, mean_fft, percentile_50, median]

    all_features = np.array(gradient_features + color_features, dtype=np.float64)

    all_features = np.nan_to_num(all_features, nan=0.0, posinf=1e5, neginf=-1e5)

    return all_features

def load_data(data_folder):
    x_data = []
    y_data = []

    # Find all .jpg files in 'camera' folder
    # Label: 0
    camera_paths = glob.glob(os.path.join(data_folder, 'camera', '*.jpg'))
    for path in camera_paths:
        features = extract_features(path)
        if features is not None:
            x_data.append(features)
            y_data.append(0)

    # Find all .jpg files in 'pizza' folder
    # Label: 1
    pizza_paths = glob.glob(os.path.join(data_folder, 'pizza', '*.jpg'))
    for path in pizza_paths:
        features = extract_features(path)
        if features is not None:
            x_data.append(features)
            y_data.append(1)

    return np.array(x_data), np.array(y_data)

def trainPerceptron(inputs, t, rho, iterNo):

    # Add bias (13 features -> 14)
    n_samples, n_features = inputs.shape
    # We add '1' to the beginning of each sample
    inputs_with_bias = np.hstack((np.ones((n_samples, 1)), inputs))

    # Initialize weights randomly
    weights = np.random.rand(n_features + 1)

    print(f"Training starting. {iterNo} iterations.")

    for i in range(iterNo):

        # Shuffle data at the beginning of each epoch
        inputs_shuffled, t_shuffled = shuffle(inputs_with_bias, t)

        # Iterate through all samples
        for x_i, t_i in zip(inputs_shuffled, t_shuffled):

            # --- Feed Forward ---
            # 1. Calculate weighted sum (dot product)
            weighted_sum = np.dot(weights, x_i)

            # 2. Get prediction by passing through Swish activation
            y = swish(weighted_sum)

            # --- Feed Backward ---

            # 1. Calculate the error
            error = t_i - y

            # 2. Calculate the derivative of Swish
            swish_grad = swish_derivative(weighted_sum)

            # 3. Calculate the weight change amount (delta w)
            # delta_w = eta * (t-y) * g'(y) * x_i
            delta_w = rho * error * swish_grad * x_i

            # 4. Update weights
            weights = weights + delta_w

        if (i+1) % 100 == 0:
            print(f"Iteration {i+1}/{iterNo} completed.")

    # Save the weights
    np.save('weights.npy', weights)
    print("Training complete. Weights saved to 'weights.npy'.")
    return weights

def testPerceptron(sample_test, weights):

    # Add bias to the test sample (as in training)
    sample_with_bias = np.insert(sample_test, 0, 1)

    # Perform feed forward (same as in training)
    weighted_sum = np.dot(weights, sample_with_bias)
    prediction_raw = swish(weighted_sum)

    # Round the output to 0 or 1
    predicted_label = 1 if prediction_raw > 0.5 else 0

    return predicted_label, prediction_raw


# --- MAIN EXECUTION ---
if __name__ == "__main__":

    # 1. LOAD DATA
    # We assume images are in "CaltechTinyTwoClass/train" and "CaltechTinyTwoClass/test"
    # I uploaded the drive
    train_folder = '/content/drive/MyDrive/CaltechTinyTwoClass/train/'
    test_folder = '/content/drive/MyDrive/CaltechTinyTwoClass/test/'

    print("Loading training data and extracting features.")
    x_train, y_train = load_data(train_folder)
    print(f"Training data loaded. (Samples, Features): {x_train.shape}")

    print("Loading test data and extracting features.")
    x_test, y_test = load_data(test_folder)
    print(f"Test data loaded. (Samples, Features): {x_test.shape}")


    # 1. Calculate mean and std dev from TRAINING data only
    train_mean = np.mean(x_train, axis=0)
    train_std = np.std(x_train, axis=0)
    epsilon = 1e-6 # To prevent division by zero if std dev is 0

    # 2. Normalize both TRAINING and TEST data using these values
    x_train = (x_train - train_mean) / (train_std + epsilon)
    x_test = (x_test - train_mean) / (train_std + epsilon)

    print("Data normalization complete.")

    # 2. TRAIN THE MODEL
    learning_rate = 0.0001 # rho value specified in PDF
    iterations = 1000      # iteration count specified in PDF

    trained_weights = trainPerceptron(x_train, y_train, learning_rate, iterations)

    # 3. TEST THE MODEL
    print("\nTest phase starting.")
    correct_predictions = 0
    results_data = [] # List to store results for the table

    for i in range(len(x_test)):
        sample = x_test[i]
        true_label = y_test[i]

        predicted_label, raw_output = testPerceptron(sample, trained_weights)

        label_map = {0: 'camera', 1: 'pizza'}
        true_label_str = label_map[true_label]
        pred_label_str = label_map[predicted_label]

        # Determine result icon
        result_icon = "✓" if predicted_label == true_label else "X"

        # Store data for the table
        results_data.append([
            i + 1,
            true_label_str,
            pred_label_str,
            f"{raw_output:.4f}",
            result_icon
        ])

        if predicted_label == true_label:
            correct_predictions += 1

    # --- Print the results table ---
    print("\n## 3.3 Detailed Test Results\n")
    # Print Header
    print("| Sample | True Label | Predicted Label | Confidence | Result |")
    print("| :--- | :--- | :--- | :--- | :--- |")

    # Print Rows
    for row in results_data:
        print(f"| {row[0]} | {row[1]} | {row[2]} | {row[3]} | {row[4]} |")

    # Calculate and print final accuracy
    accuracy = (correct_predictions / len(x_test)) * 100
    print(f"\nTest Complete.")
    print(f"Total {len(x_test)} test samples, {correct_predictions} correct predictions.")
    print(f"Test Accuracy: {accuracy:.2f}%")

Loading training data and extracting features.
Training data loaded. (Samples, Features): (82, 13)
Loading test data and extracting features.
Test data loaded. (Samples, Features): (11, 13)
Data normalization complete.
Training starting. 1000 iterations.
Iteration 100/1000 completed.
Iteration 200/1000 completed.
Iteration 300/1000 completed.
Iteration 400/1000 completed.
Iteration 500/1000 completed.
Iteration 600/1000 completed.
Iteration 700/1000 completed.
Iteration 800/1000 completed.
Iteration 900/1000 completed.
Iteration 1000/1000 completed.
Training complete. Weights saved to 'weights.npy'.

Test phase starting.

## 3.3 Detailed Test Results

| Sample | True Label | Predicted Label | Confidence | Result |
| :--- | :--- | :--- | :--- | :--- |
| 1 | camera | camera | -0.2161 | ✓ |
| 2 | camera | camera | 0.1101 | ✓ |
| 3 | camera | camera | 0.3029 | ✓ |
| 4 | camera | camera | -0.1161 | ✓ |
| 5 | camera | camera | -0.0511 | ✓ |
| 6 | pizza | camera | 0.3470 | X |
| 7 | pizza | p